# Method 4: Fine-tuning a BERT-style model on (previous reply, target reply)

**What is happening in simple words**

1. **BERT is already trained** on huge amounts of English text, so it knows grammar, slang and the meaning of many words.
   TF-IDF and a LSTM trained from scratch only know the words in *your* training file. This is the big advantage here, because your test data uses words your training data never contains.
2. **Fine-tuning** means we take that pretrained model and train it a little more on *your* labelled data so it learns your 12 classes.
   This is normal machine-learning training. It is **not** an LLM call.
3. **How the context is given to the model:** BERT can read **two sentences at once**. We give it
   `[CLS] previous_reply [SEP] target_reply [SEP]`. The model reads both together, so it can decide things like
   "this reply agrees with a toxic previous reply".
4. **Which text columns:** the **original** text (`previous_reply`, `target_reply`), **not** the `*_clean` columns.
   BERT's tokenizer already handles lowercase, punctuation and emojis, and it uses punctuation and capital letters as clues.
   So we skip the cleaning from the preprocessing notebook here.

**Plan of this notebook**

| Step | What |
|---|---|
| 1 | Settings (everything you may want to change is in one cell) |
| 2 | Load data and check labels |
| 3 | Split off a validation set (same group-safe method as before) |
| 4 | Tokenize (turn text into numbers) |
| 5 | Build the model |
| 6 | Train with early stopping |
| 7 | Evaluate on validation and on the separate test set |
| 8 | Save the probabilities (for the ensemble later) |

**Run it on a GPU:** in Google Colab use *Runtime → Change runtime type → T4 GPU*. On a CPU it will be very slow.

## Step 0: Install and import

In [1]:
# Colab already has torch. If something is missing, run this once:
# !pip install -q transformers scikit-learn pandas numpy tqdm

import random, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from transformers import (AutoTokenizer, AutoModel, DataCollatorWithPadding,
                          get_linear_schedule_with_warmup)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## Step 1: Settings

Everything you might want to change is here.

* **`MODEL_NAME`** – your data is English only, so `bert-base-uncased` is a good default.
  Faster/smaller: `distilbert-base-uncased`. Often stronger on social-media text: `roberta-base`.
* **`TOXIC_LOSS_WEIGHT`** – the *optional multi-task* idea. The model gets a second small job: guess "toxic or not".
  Learning both together often helps the small classes. Set it to `0.0` to turn this off and compare.
* **`MAX_LEN`** – maximum number of tokens for both sentences together. Longer text is cut. We check below that very little is cut.

In [4]:
# ---------- files ----------
DEV_FILE  = "../preprocessing/train_data_with_preprocessing1.csv"             #  Train data (train + val together)
TEST_FILE = "../preprocessing/test_data_with_preprocessing1.csv"       # separate test data

# ORIGINAL text columns (NOT the *_clean ones)
DEV_PREV,  DEV_TARGET  = "previous_reply", "target_reply"
TEST_PREV, TEST_TARGET = "context", "reply"          # the English translation columns of test.csv

LABEL_COL = "label_id"      # number column with the class (0..11)
GROUP_COL = "group"         # same target sentence = same group (from the preprocessing notebook)

# ---------- model ----------
MODEL_NAME = "bert-base-uncased"
MAX_LEN    = 128

# ---------- training ----------
BATCH_SIZE        = 32
LEARNING_RATE     = 2e-5
EPOCHS            = 5          # maximum; early stopping may stop sooner
WARMUP_FRACTION   = 0.1        # first 10% of steps use a small learning rate that slowly grows
WEIGHT_DECAY      = 0.01
PATIENCE          = 2          # stop if validation macro-F1 does not improve for 2 epochs
TOXIC_LOSS_WEIGHT = 0.0       # 0.0 = only the 12-class task, 0.5 = also learn "toxic or not"
SEED              = 42

use_fp16 = (device.type == "cuda")   # half precision = faster + less memory on a GPU

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

## Step 2: Load the data

Important: we read with `keep_default_na=False`. Otherwise pandas turns the text label `"None"` into a missing value.

In [5]:
dev_df  = pd.read_csv(DEV_FILE,  keep_default_na=False).reset_index(drop=True)
test_df = pd.read_csv(TEST_FILE, keep_default_na=False).reset_index(drop=True)

# Check that the columns exist - if not, the message tells you what to change in the settings cell
for df, cols, name in [(dev_df,  [DEV_PREV, DEV_TARGET, LABEL_COL, GROUP_COL], "DEV_FILE"),
                       (test_df, [TEST_PREV, TEST_TARGET, LABEL_COL],          "TEST_FILE")]:
    missing = [c for c in cols if c not in df.columns]
    assert not missing, f"{name} is missing columns {missing}. Columns found: {list(df.columns)}"

dev_df[LABEL_COL]  = dev_df[LABEL_COL].astype(int)
test_df[LABEL_COL] = test_df[LABEL_COL].astype(int)

N_CLASSES = int(dev_df[LABEL_COL].max()) + 1
print("Development rows:", len(dev_df), "| Test rows:", len(test_df), "| Classes:", N_CLASSES)

# Class names (only if a text 'label' column exists), so reports are readable
if "label" in dev_df.columns:
    id_to_name = dict(zip(dev_df[LABEL_COL], dev_df["label"]))
else:
    id_to_name = {i: str(i) for i in range(N_CLASSES)}
class_names = [id_to_name[i] for i in range(N_CLASSES)]
NONE_ID = class_names.index("None") if "None" in class_names else 0   # the 'not toxic' class
print("Class ids:", {i: n for i, n in enumerate(class_names)})
print("'Not toxic' class id:", NONE_ID)

dev_df[[DEV_PREV, DEV_TARGET, LABEL_COL]].head()

Development rows: 19431 | Test rows: 1804 | Classes: 12
Class ids: {0: 'None', 1: 'Dehumanization', 2: 'Harassment', 3: 'Hate Speech', 4: 'Identity Attack', 5: 'Incitement / Encouragement of Harm', 6: 'Insult', 7: 'Mockery / Ridicule', 8: 'Profanity', 9: 'Sexual Harassment / Objectification', 10: 'Threat', 11: 'Toxic Endorsement / Agreement'}
'Not toxic' class id: 0


,previous_reply,target_reply,label_id
0,I shared photos from today's event.,"In this video, your body draws more attention ...",9
1,I see no reason to change my opinion.,"Hearing this statement, it seems imagination d...",7
2,"Someone in the discussion said, There is no ne...",Not everyone is the same; people should be tre...,0
3,What's the update on today's work?,"So far, More details can be given later.",0
4,I only disagreed politely.,"I'm warning you, if you ask this question agai...",10


## Step 3: Split off a validation set (group-safe)

Same idea as in the preprocessing notebook: all rows with the same target sentence (same `group`) stay together,
so the validation score is not inflated by memorized sentences.

Fine-tuning BERT is slow, so we do **not** do 10-fold cross-validation here. We use one of the 10 groups as validation (about 10%) and the rest for training.

In [6]:
splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
train_idx, val_idx = next(splitter.split(dev_df, dev_df[LABEL_COL], dev_df[GROUP_COL]))

train_df = dev_df.iloc[train_idx].reset_index(drop=True)
val_df   = dev_df.iloc[val_idx].reset_index(drop=True)

print("Train rows:", len(train_df), "| Validation rows:", len(val_df))
print("Target sentences shared between train and val:",
      len(set(train_df[GROUP_COL]) & set(val_df[GROUP_COL])), "(must be 0)")

Train rows: 17544 | Validation rows: 1887
Target sentences shared between train and val: 0 (must be 0)


## Step 4: Tokenize

The tokenizer turns text into numbers (token ids). We give it **both** sentences: `tokenizer(previous, target)`.
It builds `[CLS] previous [SEP] target [SEP]` for us.

* Texts are **not** padded here. Each batch is padded to its own longest text later (`DataCollatorWithPadding`), which is faster.
* `truncation=True` cuts texts that are longer than `MAX_LEN`.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Example of what the model sees
example = tokenizer(train_df[DEV_PREV][0], train_df[DEV_TARGET][0], truncation=True, max_length=MAX_LEN)
print(tokenizer.decode(example["input_ids"]))

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

c:\Users\rajde\anaconda3\envs\pytrch310\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rajde\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[CLS] i shared photos from today ' s event. [SEP] in this video, your body draws more attention than the content. [SEP]


In [8]:
# How long are the texts? We want almost no text to be cut at MAX_LEN.
def token_lengths(prev, target):
    enc = tokenizer(list(prev), list(target), truncation=False)
    return np.array([len(x) for x in enc["input_ids"]])

for name, df, p, t in [("train", train_df, DEV_PREV, DEV_TARGET), ("test", test_df, TEST_PREV, TEST_TARGET)]:
    lens = token_lengths(df[p], df[t])
    print(f"{name:5s}: median {np.median(lens):.0f} tokens | 95th percentile {np.percentile(lens,95):.0f} | "
          f"longer than MAX_LEN={MAX_LEN}: {100*(lens > MAX_LEN).mean():.1f}%")

train: median 32 tokens | 95th percentile 45 | longer than MAX_LEN=128: 0.0%
test : median 33 tokens | 95th percentile 73 | longer than MAX_LEN=128: 0.7%


In [9]:
class PairDataset(Dataset):
    """Holds the tokenized (previous, target) pairs and, if given, the labels."""
    def __init__(self, prev_texts, target_texts, labels=None):
        self.enc = tokenizer([str(x) for x in prev_texts], [str(x) for x in target_texts],
                             truncation=True, max_length=MAX_LEN)
        self.keys = [k for k in ("input_ids", "attention_mask", "token_type_ids") if k in self.enc]
        self.labels = None if labels is None else [int(x) for x in labels]

    def __len__(self):
        return len(self.enc["input_ids"])

    def __getitem__(self, i):
        item = {k: self.enc[k][i] for k in self.keys}
        if self.labels is not None:
            item["labels"] = self.labels[i]
        return item

collator = DataCollatorWithPadding(tokenizer)       # pads each batch to its longest text

train_ds = PairDataset(train_df[DEV_PREV],  train_df[DEV_TARGET],  train_df[LABEL_COL])
val_ds   = PairDataset(val_df[DEV_PREV],    val_df[DEV_TARGET],    val_df[LABEL_COL])
test_ds  = PairDataset(test_df[TEST_PREV],  test_df[TEST_TARGET],  test_df[LABEL_COL])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,     shuffle=True,  collate_fn=collator)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator)
print("Batches per epoch:", len(train_loader))

Batches per epoch: 549


## Step 5: Build the model

The model has 3 parts:
1. **The pretrained BERT encoder** reads the sentence pair and gives one vector for the `[CLS]` position, a summary of both sentences.
2. **Type head**: one linear layer that turns this summary into 12 scores (one per toxicity type).
3. **Toxic head** (optional extra task): one linear layer that gives 1 score for "toxic or not".

**Class imbalance:** None is about half of the data while small classes are about 3-4%.
We give each class a **weight** (rare class = bigger weight) so mistakes on small classes cost more.

In [10]:
class ToxicityModel(nn.Module):
    def __init__(self, model_name, n_classes, dropout=0.1):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        hidden          = self.encoder.config.hidden_size
        self.dropout    = nn.Dropout(dropout)
        self.type_head  = nn.Linear(hidden, n_classes)   # 12-class scores
        self.toxic_head = nn.Linear(hidden, 1)           # toxic / not toxic score

    def forward(self, **inputs):
        summary = self.encoder(**inputs).last_hidden_state[:, 0]    # vector at the [CLS] position
        summary = self.dropout(summary)
        return self.type_head(summary), self.toxic_head(summary).squeeze(-1)

model = ToxicityModel(MODEL_NAME, N_CLASSES).to(device)

# class weights: rare classes get a bigger weight
weights = compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=train_df[LABEL_COL].values)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)
print({class_names[i]: round(float(w), 2) for i, w in enumerate(weights)})

type_loss_fn  = nn.CrossEntropyLoss(weight=class_weights)
toxic_loss_fn = nn.BCEWithLogitsLoss()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

{'None': 0.16, 'Dehumanization': 2.05, 'Harassment': 3.0, 'Hate Speech': 1.93, 'Identity Attack': 2.04, 'Incitement / Encouragement of Harm': 2.04, 'Insult': 0.85, 'Mockery / Ridicule': 1.93, 'Profanity': 2.11, 'Sexual Harassment / Objectification': 2.13, 'Threat': 1.83, 'Toxic Endorsement / Agreement': 2.56}


## Step 6: Train

For each batch the model: makes predictions → we measure the error (loss) → we update the weights a little.

* **AdamW + warmup + linear decay**: the learning rate starts tiny, rises for the first 10% of steps, then slowly falls to 0. This is the standard, stable way to fine-tune BERT.
* **Gradient clipping**: stops a single bad batch from changing the model too much.
* **fp16**: on a GPU the maths runs in half precision, which is faster.
* **Early stopping**: after every epoch we measure **macro-F1 on validation** (macro-F1 treats all 12 classes equally, so small classes matter).
  We keep the best version of the model and stop when it has not improved for `PATIENCE` epochs.

In [11]:
optimizer   = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRACTION * total_steps), total_steps)
scaler      = torch.amp.GradScaler("cuda", enabled=use_fp16)


@torch.no_grad()
def predict_probs(model, loader):
    """Returns an array (n_rows, n_classes) with the probability of every class."""
    model.eval()
    all_logits = []
    for batch in loader:
        inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_fp16):
            type_logits, _ = model(**inputs)
        all_logits.append(type_logits.float().cpu())
    return torch.softmax(torch.cat(all_logits), dim=1).numpy()


def score(y_true, probs):
    pred = probs.argmax(axis=1)
    return {"macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
            "accuracy": accuracy_score(y_true, pred),
            "toxic_f1": f1_score(np.array(y_true) != NONE_ID, pred != NONE_ID)}   # toxic vs not toxic

In [12]:
best_f1, best_state, bad_epochs = -1.0, None, 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, start = 0.0, time.time()

    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
        labels = batch.pop("labels").to(device)
        inputs = {k: v.to(device) for k, v in batch.items()}

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_fp16):
            type_logits, toxic_logit = model(**inputs)
            loss = type_loss_fn(type_logits.float(), labels)
            if TOXIC_LOSS_WEIGHT > 0:                              # optional second task
                is_toxic = (labels != NONE_ID).float()
                loss = loss + TOXIC_LOSS_WEIGHT * toxic_loss_fn(toxic_logit.float(), is_toxic)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item()

    val_scores = score(val_df[LABEL_COL].values, predict_probs(model, val_loader))
    history.append({"epoch": epoch, "train_loss": running_loss / len(train_loader), **val_scores})
    print(f"Epoch {epoch}: train loss {running_loss/len(train_loader):.4f} | "
          f"val macro-F1 {val_scores['macro_f1']:.4f} | val acc {val_scores['accuracy']:.4f} | "
          f"toxic-vs-not F1 {val_scores['toxic_f1']:.4f} | {time.time()-start:.0f}s")

    if val_scores["macro_f1"] > best_f1:                           # new best -> remember it
        best_f1, bad_epochs = val_scores["macro_f1"], 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("Early stopping: no improvement for", PATIENCE, "epochs.")
            break

model.load_state_dict(best_state)                                  # go back to the best epoch
print("\nBest validation macro-F1:", round(best_f1, 4))
pd.DataFrame(history)

Epoch 1/5:   0%|          | 0/549 [00:00<?, ?it/s]

Epoch 1: train loss 0.6653 | val macro-F1 0.9562 | val acc 0.9746 | toxic-vs-not F1 0.9984 | 98s


Epoch 2/5:   0%|          | 0/549 [00:00<?, ?it/s]

Epoch 2: train loss 0.0675 | val macro-F1 0.9646 | val acc 0.9783 | toxic-vs-not F1 0.9984 | 97s


Epoch 3/5:   0%|          | 0/549 [00:00<?, ?it/s]

Epoch 3: train loss 0.0269 | val macro-F1 0.9693 | val acc 0.9820 | toxic-vs-not F1 0.9984 | 97s


Epoch 4/5:   0%|          | 0/549 [00:00<?, ?it/s]

Epoch 4: train loss 0.0109 | val macro-F1 0.9682 | val acc 0.9820 | toxic-vs-not F1 0.9990 | 97s


Epoch 5/5:   0%|          | 0/549 [00:00<?, ?it/s]

Epoch 5: train loss 0.0041 | val macro-F1 0.9667 | val acc 0.9809 | toxic-vs-not F1 0.9984 | 97s
Early stopping: no improvement for 2 epochs.

Best validation macro-F1: 0.9693


,epoch,train_loss,macro_f1,accuracy,toxic_f1
0,1,0.665289,0.956186,0.974563,0.998430
1,2,0.067491,0.964620,0.978272,0.998430
2,3,0.026943,0.969342,0.981982,0.998428
3,4,0.010877,0.968214,0.981982,0.998953
4,5,0.004142,0.966654,0.980922,0.998428


## Step 7a: Validation results (same kind of text as training)

Look at the **per-class** F1 and recall, not only the average. If a small class has recall near 0, the model is ignoring it.

In [13]:
val_probs = predict_probs(model, val_loader)
val_pred  = val_probs.argmax(axis=1)
print(classification_report(val_df[LABEL_COL], val_pred, labels=list(range(N_CLASSES)),
                            target_names=class_names, digits=3, zero_division=0))

                                     precision    recall  f1-score   support

                               None      0.999     0.998     0.998       933
                     Dehumanization      0.935     0.986     0.960        73
                         Harassment      0.985     1.000     0.993        67
                        Hate Speech      0.902     0.902     0.902        51
                    Identity Attack      1.000     1.000     1.000        82
 Incitement / Encouragement of Harm      0.988     0.976     0.982        83
                             Insult      0.958     0.934     0.946       197
                 Mockery / Ridicule      1.000     1.000     1.000        87
                          Profanity      0.944     0.977     0.960        86
Sexual Harassment / Objectification      0.931     0.882     0.905        76
                             Threat      0.972     1.000     0.986       105
      Toxic Endorsement / Agreement      1.000     1.000     1.000        4

## Step 7b: Test results (new, different kind of text)

This is the real-world check. **Do not tune anything on this** (not `MAX_LEN`, not the learning rate...). Look at it, report it.
Many test classes have very few rows (look at the `support` column), so their F1 scores are noisy.

In [14]:
test_probs = predict_probs(model, test_loader)
test_pred  = test_probs.argmax(axis=1)
y_test     = test_df[LABEL_COL].values

s = score(y_test, test_probs)
print(f"TEST  macro-F1: {s['macro_f1']:.4f} | accuracy: {s['accuracy']:.4f} | toxic-vs-not F1: {s['toxic_f1']:.4f}\n")
print(classification_report(y_test, test_pred, labels=list(range(N_CLASSES)),
                            target_names=class_names, digits=3, zero_division=0))

TEST  macro-F1: 0.1804 | accuracy: 0.3769 | toxic-vs-not F1: 0.7163

                                     precision    recall  f1-score   support

                               None      0.629     0.473     0.540       786
                     Dehumanization      0.196     0.450     0.273        20
                         Harassment      1.000     0.011     0.022        88
                        Hate Speech      0.101     0.577     0.172        26
                    Identity Attack      0.103     0.308     0.154        13
 Incitement / Encouragement of Harm      0.100     0.048     0.065        21
                             Insult      0.340     0.671     0.451       346
                 Mockery / Ridicule      0.331     0.107     0.161       375
                          Profanity      0.125     0.053     0.074        57
Sexual Harassment / Objectification      0.000     0.000     0.000        55
                             Threat      0.500     0.143     0.222         7
      

In [15]:
# Which classes get confused? Rows = true class, columns = predicted class.
short_names = [n[:10] for n in class_names]
cm = confusion_matrix(y_test, test_pred, labels=list(range(N_CLASSES)))
pd.DataFrame(cm, index=short_names, columns=short_names)

,None,Dehumaniza,Harassment,Hate Speec,Identity A,Incitement,Insult,Mockery /,Profanity,Sexual Har,Threat,Toxic Endo
None,372,11,0,37,13,5,177,48,6,4,1,112
Dehumaniza,2,9,0,3,0,0,6,0,0,0,0,0
Harassment,17,0,1,8,7,0,50,4,0,1,0,0
Hate Speec,1,5,0,15,0,0,4,1,0,0,0,0
Identity A,0,1,0,5,4,0,3,0,0,0,0,0
Incitement,5,3,0,1,2,1,7,1,1,0,0,0
Insult,48,4,0,39,4,1,232,11,4,1,0,2
Mockery /,107,10,0,28,8,3,155,40,10,5,0,9
Profanity,12,3,0,5,0,0,22,9,3,1,0,2
Sexual Har,21,0,0,6,1,0,21,5,0,0,0,1


In [16]:
# How sure can we be about the test macro-F1? Re-sample the test rows 300 times and look at the spread.
rng = np.random.default_rng(SEED)
scores = []
for _ in range(300):
    pick = rng.integers(0, len(y_test), len(y_test))
    scores.append(f1_score(y_test[pick], test_pred[pick], average="macro", zero_division=0))
print(f"Test macro-F1 = {np.mean(scores):.3f}  (95% range: {np.percentile(scores, 2.5):.3f} to {np.percentile(scores, 97.5):.3f})")

Test macro-F1 = 0.177  (95% range: 0.142 to 0.217)


## Step 8: Save everything

The probability files are what the **ensemble** needs later (each method saves the same kind of file: one row per example, one column per class).

In [ ]:
np.save("bert_val_probs.npy",  val_probs)
np.save("bert_test_probs.npy", test_probs)
np.save("bert_val_row_index.npy", val_idx)            # which rows of dev_df are the validation rows

out = test_df[[TEST_PREV, TEST_TARGET, LABEL_COL]].copy()
out["true_label"] = [class_names[i] for i in out[LABEL_COL]]
out["pred_label"] = [class_names[i] for i in test_pred]
out["confidence"] = test_probs.max(axis=1).round(3)
out.to_csv("bert_test_predictions.csv", index=False)

json.dump({"model": MODEL_NAME, "max_len": MAX_LEN, "toxic_loss_weight": TOXIC_LOSS_WEIGHT,
           "best_val_macro_f1": best_f1, "class_names": class_names},
          open("bert_run_info.json", "w"), indent=2)

torch.save(model.state_dict(), "bert_best_model.pt")      # (large file, ~400 MB)
print("Saved: bert_val_probs.npy, bert_test_probs.npy, bert_test_predictions.csv, bert_run_info.json, bert_best_model.pt")

## Things to try next (one change at a time, compare validation macro-F1)

| Experiment | How |
|---|---|
| Is the extra "toxic or not" task helping? | Run once with `TOXIC_LOSS_WEIGHT = 0.5` and once with `0.0` |
| Different model | `MODEL_NAME = "roberta-base"` or `"distilbert-base-uncased"` |
| Is the context helping? | Replace the previous reply with an empty string and compare |
| Less over-fitting to templates | Fewer epochs, or a lower learning rate (`1e-5`) |
| More stable score | Repeat with 3 different `SEED` values and average |

**Reading the results:** if the validation score is very high (like 0.95+) but the test score is low, the model has learned your training templates, not toxicity in general.
That is a property of the training data, not a bug. Report both numbers.